# California County Spending Efficiency

## Karen Wu

### Business Question

Which California counties are allocating budget most efficiently relative to population size and service demand, and where are there notable spending trends over time?

In [3]:
import pandas as pd

# Load each file - adjust filenames to match exactly what you downloaded
revenues = pd.read_csv('data/county_revenues.csv')
expenditures = pd.read_csv('data/county_expenditures.csv')
revenues_per_capita = pd.read_csv('data/county_revenues_per_capita.csv')
expenditures_per_capita = pd.read_csv('data/county_expenditures_per_capita.csv')

# Inspect each one
for name, df in [('Revenues', revenues), ('Expenditures', expenditures), 
                  ('Revenues Per Capita', revenues_per_capita), 
                  ('Expenditures Per Capita', expenditures_per_capita)]:
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(df.head(3))


Revenues
Shape: (581343, 15)
Columns: ['Entity Name', 'Fiscal Year', 'Type', 'Form/Table', 'Category', 'Subcategory 1', 'Subcategory 2', 'Subcategory 3', 'Subcategory 4', 'Line Description', 'Values', 'Zip Code', 'Estimated Population', 'Area in Sq. Miles', 'Row Number']
  Entity Name  Fiscal Year      Type   Form/Table               Category  \
0     Alameda         2017  Revenues   CHRGS_SERV  Internal Service Fund   
1     Alameda         2017  Revenues  OTHR_OP_REV  Internal Service Fund   
2     Alameda         2017  Revenues  INVEST_EARN  Internal Service Fund   

           Subcategory 1                                   Subcategory 2  \
0     Operating Revenues      Charges for Services_Internal Service Fund   
1     Operating Revenues  Other Operating Revenues_Internal Service Fund   
2  Nonoperating Revenues       Investment Earnings_Internal Service Fund   

                                    Subcategory 3  \
0      Charges for Services_Internal Service Fund   
1  Other Op

### Merge the two per-capita datasets on county and fiscal year

Since these files are pre-aggregated, it is easy to merge and will allow us to build the core "spending efficiency" datasets


In [4]:
# Merge the two per-capita datasets on county and fiscal year
spending_efficiency = revenues_per_capita.merge(
    expenditures_per_capita,
    on=['Entity Name', 'Fiscal Year', 'Estimated Population'],
    how='inner'
)

# Calculate a simple efficiency metric: surplus/deficit per capita
spending_efficiency['Net Per Capita'] = (
    spending_efficiency['Revenues Per Capita'] - spending_efficiency['Expenditures Per Capita']
)

print(spending_efficiency.shape)
spending_efficiency.head(10)

(1254, 8)


,Entity Name,Fiscal Year,Total Revenues,Estimated Population,Revenues Per Capita,Total Expenditures,Expenditures Per Capita,Net Per Capita
0,Alameda,2024,4742901769,1641869,2889,4244700272,2585,304
1,Alpine,2024,30191475,1179,25608,27326942,23178,2430
2,Amador,2024,144198400,39611,3640,125814417,3176,464
3,Butte,2024,657810470,205928,3194,604671332,2936,258
4,Calaveras,2024,187146424,44842,4173,163584784,3648,525
5,Colusa,2024,92900505,21743,4273,92032288,4233,40
6,Contra Costa,2024,5970212101,1146626,5207,5549211078,4840,367
7,Del Norte,2024,115988068,26345,4403,98317911,3732,671
8,El Dorado,2024,629861635,188583,3340,586097683,3108,232
9,Fresno,2024,2739998180,1017431,2693,2636081403,2591,102


In [5]:
# Filter to fiscal year 2024 only
fy2024 = spending_efficiency[spending_efficiency['Fiscal Year'] == 2024].copy()

# Sort by Net Per Capita to see who's running surplus vs deficit
fy2024_sorted = fy2024.sort_values('Net Per Capita', ascending=False)

print(f"Number of counties: {fy2024_sorted.shape[0]}")
print(f"\nTop 5 counties by Net Per Capita (surplus):")
print(fy2024_sorted[['Entity Name', 'Revenues Per Capita', 'Expenditures Per Capita', 'Net Per Capita']].head(5))

print(f"\nBottom 5 counties by Net Per Capita (deficit):")
print(fy2024_sorted[['Entity Name', 'Revenues Per Capita', 'Expenditures Per Capita', 'Net Per Capita']].tail(5))

Number of counties: 57

Top 5 counties by Net Per Capita (surplus):
   Entity Name  Revenues Per Capita  Expenditures Per Capita  Net Per Capita
1       Alpine                25608                    23178            2430
51     Trinity                 6001                     5276             725
7    Del Norte                 4403                     3732             671
13        Inyo                 6803                     6146             657
4    Calaveras                 4173                     3648             525

Bottom 5 counties by Net Per Capita (deficit):
    Entity Name  Revenues Per Capita  Expenditures Per Capita  Net Per Capita
10        Glenn                 5027                     5087             -60
15        Kings                 2672                     2826            -154
41  Santa Clara                 5465                     5634            -169
27         Napa                 4362                     4567            -205
50       Tehama                 

### Possible outliers

Some counties are very small like Alpine (~1200), which inflates the per capita numbers since the denominator(population) is tiny.

Alpine has ~1,200 people, so its $2,430 per capita "surplus" represents a relatively small absolute dollar amount spread over a tiny population, while a huge, dense county like Santa Clara is showing a deficit that's actually a rounding error relative to its scale.

# Population tiers

Let's add a column and categorize a county into the appropriate population for a more contextualized analysis.

- Small counties: under 50k
- Medium counties: 50k - 250k
- Large counties: 250k+

In [9]:
# Add population tiers to contextualize per capita figures
def population_tier(pop):
    if pop < 50000:
        return 'Small (under 50K)'
    elif pop < 250000:
        return 'Medium (50K-250K)'
    else:
        return 'Large (250K+)'

fy2024_sorted['Population Tier'] = fy2024_sorted['Estimated Population'].apply(population_tier)

# Check the tier breakdown
print(fy2024_sorted['Population Tier'].value_counts())

# Now look at top/bottom WITHIN each tier for a fairer comparison
for tier in ['Large (250K+)', 'Medium (50K-250K)', 'Small (under 50K)']:
    tier_data = fy2024_sorted[fy2024_sorted['Population Tier'] == tier]
    print(f"\n{'='*50}")
    print(f"{tier} — {len(tier_data)} counties")
    print(f"{'='*50}")
    print(tier_data[['Entity Name', 'Revenues Per Capita', 'Expenditures Per Capita', 'Net Per Capita']].head(3))

Population Tier
Large (250K+)        25
Medium (50K-250K)    17
Small (under 50K)    15
Name: count, dtype: int64

Large (250K+) — 25 counties
       Entity Name  Revenues Per Capita  Expenditures Per Capita  \
39       San Mateo                 4117                     3695   
6     Contra Costa                 5207                     4840   
35  San Bernardino                 3337                     2993   

    Net Per Capita  
39             422  
6              367  
35             344  

Medium (50K-250K) — 17 counties
   Entity Name  Revenues Per Capita  Expenditures Per Capita  Net Per Capita
19      Madera                 3068                     2762             306
3        Butte                 3194                     2936             258
8    El Dorado                 3340                     3108             232

Small (under 50K) — 15 counties
   Entity Name  Revenues Per Capita  Expenditures Per Capita  Net Per Capita
1       Alpine                25608              

# Why the largest counties aren't in the top 3

Large counties like LA, San Diego, and Orange likely have far more complex service obligations — social services, public health systems, court systems, jails — that scale faster than population alone. 

Smaller counties often have leaner service models, so a smaller gap between revenue and spending isn't surprising.

## Why this is a stronger story than "biggest counties spend the most":

Most people would assume bigger = less efficient or bigger = more strained. The data shows the opposite — Contra Costa and San Mateo are running healthy surpluses despite being large, dense counties. That's a finding worth highlighting prominently.

In [7]:
# Average net per capita by population tier
tier_summary = fy2024_sorted.groupby('Population Tier').agg(
    avg_revenue_per_capita=('Revenues Per Capita', 'mean'),
    avg_expenditure_per_capita=('Expenditures Per Capita', 'mean'),
    avg_net_per_capita=('Net Per Capita', 'mean'),
    county_count=('Entity Name', 'count')
).round(0)

print(tier_summary)

                   avg_revenue_per_capita  avg_expenditure_per_capita  \
Population Tier                                                         
Large (250K+)                      3389.0                      3236.0   
Medium (50K-250K)                  3420.0                      3373.0   
Small (under 50K)                  7049.0                      6563.0   

                   avg_net_per_capita  county_count  
Population Tier                                      
Large (250K+)                   153.0            25  
Medium (50K-250K)                47.0            17  
Small (under 50K)               486.0            15  


## What is the data telling us:

- Small counties have by far the highest per capita revenue and expenditure (7,049 vs ~3,400 for the other tiers); this is likely driven by Alpine's extreme skew (25,608 per capita) pulling up the small-county average
- Large counties actually have a healthier average surplus of 153 compared to medium counties surplus of 47; so my initial instinct about large counties not being worse off holds up
- Medium counties are running the thinnest margins

### One thing worth checking before finalizing this narrative: 
Alpine is likely a major outlier dragging the "Small" tier average upward. Let's verify:

In [8]:
# Check if Alpine is skewing the small county average
small_counties = fy2024_sorted[fy2024_sorted['Population Tier'] == 'Small (under 50K)']
print(small_counties[['Entity Name', 'Revenues Per Capita', 'Net Per Capita']].sort_values('Net Per Capita', ascending=False))

# Recalculate small county average WITHOUT Alpine
small_no_alpine = small_counties[small_counties['Entity Name'] != 'Alpine']
print(f"\nSmall county avg net per capita WITHOUT Alpine: {small_no_alpine['Net Per Capita'].mean():.0f}")
print(f"Small county avg net per capita WITH Alpine: {small_counties['Net Per Capita'].mean():.0f}")

   Entity Name  Revenues Per Capita  Net Per Capita
1       Alpine                25608            2430
51     Trinity                 6001             725
7    Del Norte                 4403             671
13        Inyo                 6803             657
4    Calaveras                 4173             525
31      Plumas                 5443             498
44      Sierra                13210             480
2       Amador                 3640             464
24       Modoc                 5294             275
25        Mono                 7804             215
45    Siskiyou                 3959             197
17      Lassen                 3093             108
21    Mariposa                 7003              67
5       Colusa                 4273              40
10       Glenn                 5027             -60

Small county avg net per capita WITHOUT Alpine: 347
Small county avg net per capita WITH Alpine: 486


So excluding Alpine County's extreme population skew, small counties (under 50K population) average the highest per capita budget surplus at 347 dollars, followed by large counties (250K+) at 153 dollars, with medium counties (50K-250K) running the thinnest margins at just 47 dollars per capita.

Note that Alpine is meaningfully skewing the average (486 with it vs 347 without), though even excluding Alpine, small counties still lead with the highest average surplus across all three population tiers.

Notice that Sierra County is at $13,210 per capita revenue which is another small, low-population outlier worth being aware of, though it doesn't skew the average nearly as dramatically as Alpine.

In [11]:
# Export Results to CSV

# Export the cleaned, tiered dataset for Tableau
fy2024_sorted.to_csv('data/county_spending_efficiency_2024.csv', index=False)
print("Exported successfully")
print(fy2024_sorted.columns.tolist())

Exported successfully
['Entity Name', 'Fiscal Year', 'Total Revenues', 'Estimated Population', 'Revenues Per Capita', 'Total Expenditures', 'Expenditures Per Capita', 'Net Per Capita', 'Population Tier']
